# Day 6: Session 6B - Long and Wide

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6b_reshaping_data.html)

Date: 09/08/2026

In [1]:
import pandas as pd

base = 'https://eds-217-essential-python.github.io/data/'

goleta = pd.read_csv(base + 'openaq_goleta_measurments.csv')
santa_barbara = pd.read_csv(base + 'openaq_santa_barbara_measurments.csv')
cnsi = pd.read_csv(base + 'openaq_CNSI_measurments.csv')

print(goleta.shape)
print(santa_barbara.shape)
print(cnsi.shape)

(1962, 15)
(2266, 15)
(714, 15)


In [2]:
aq = pd.concat([goleta, santa_barbara, cnsi])

aq.shape

(4942, 15)

In [ ]:
goleta['site'] = 'Goleta'
santa_barbara['site'] = 'Santa Barbara'
cnsi['site'] = 'CNSI'

aq = pd.concat([goleta, santa_barbara, cnsi], ignore_index=True)

# ignore_index is useful because each input table has its own index, 
# numbered from zero, and stacking keeps all three of those indexes.
# Using ignore_index=True discards the old numbering and 
# renumbers the result from 0 to 4,941.

aq.shape

(4942, 16)

### test your knowledge

Two things, both of them the stacking pattern written from scratch:

1. Stack the three files again with ignore_index=True, in a different order: CNSI   first, then Goleta, then Santa Barbara. Confirm the shape is identical. Then use .head() to show that the row order changed and .value_counts() on site to show that the contents did not.

2. Stack only two of them, Goleta and CNSI, into a table called two_sites. Predict the row count out loud before you run it, then check.

In [8]:
aq_new = pd.concat([cnsi, goleta, santa_barbara], ignore_index=True)
print(aq_new.shape)

(4942, 16)


In [9]:
aq_new.head()

,location_id,location_name,parameter,value,unit,datetimeUtc,datetimeLocal,timezone,latitude,longitude,country_iso,isMobile,isMonitor,owner_name,provider,site
0,2812592,CNSI Roof Top,pm25,4.4,µg/m³,2024-07-11T01:00:00+00:00,2024-07-10T18:00:00-07:00,America/Los_Angeles,34.41556,-119.84025,NaN,NaN,NaN,Clarity,Clarity,CNSI
1,2812592,CNSI Roof Top,pm25,4.4,µg/m³,2024-07-11T02:00:00+00:00,2024-07-10T19:00:00-07:00,America/Los_Angeles,34.41556,-119.84025,NaN,NaN,NaN,Clarity,Clarity,CNSI
2,2812592,CNSI Roof Top,pm25,4.6,µg/m³,2024-07-11T04:00:00+00:00,2024-07-10T21:00:00-07:00,America/Los_Angeles,34.41556,-119.84025,NaN,NaN,NaN,Clarity,Clarity,CNSI
3,2812592,CNSI Roof Top,pm25,4.2,µg/m³,2024-07-11T05:00:00+00:00,2024-07-10T22:00:00-07:00,America/Los_Angeles,34.41556,-119.84025,NaN,NaN,NaN,Clarity,Clarity,CNSI
4,2812592,CNSI Roof Top,pm25,4.5,µg/m³,2024-07-11T06:00:00+00:00,2024-07-10T23:00:00-07:00,America/Los_Angeles,34.41556,-119.84025,NaN,NaN,NaN,Clarity,Clarity,CNSI


In [10]:
aq.value_counts('site')

site
Santa Barbara    2266
Goleta           1962
CNSI              714
Name: count, dtype: int64

In [12]:
aq_two = pd.concat([goleta, cnsi], ignore_index=True)
aq_two.shape


(2676, 16)

### Long and wide

In [13]:
aq[['site', 'parameter', 'value', 'unit', 'datetimeLocal']].head()

,site,parameter,value,unit,datetimeLocal
0,Goleta,o3,0.025,ppm,2024-07-11T18:00:00-07:00
1,Goleta,o3,0.028,ppm,2024-07-11T19:00:00-07:00
2,Goleta,o3,0.029,ppm,2024-07-11T20:00:00-07:00
3,Goleta,o3,0.027,ppm,2024-07-11T21:00:00-07:00
4,Goleta,o3,0.026,ppm,2024-07-11T22:00:00-07:00


Almost every instrument database and API gives data in long layout. However, a long table is harder to compare across than a wide table

### The pivot pattern



In [ ]:
means = aq.pivot_table(index='site', columns='parameter', values='value')

means

# pivot_table() rotates a dataframe from long to wide

# df.pivot_table(index='rows', columns='columns', values='numbers')
#                    ↑             ↑                  ↑
#               | what labels |  what labels  |   what goes     |
#               | the rows    |  the columns  |   in the cells  |

parameter,o3,pm10,pm25
site,,,
CNSI,NaN,NaN,6.083473
Goleta,0.022470,14.972921,6.480926
Santa Barbara,0.019822,17.563969,6.172324


### What happens when a cell has more than one number

pivot_table has to reduce all readings to a single number. By default it takes the mean, but we can ask for something else with 'aggfunc='

pivot_table with aggfunc= is the split-apply-combine pattern in a different arrangement!

aq.groupby(['site', 'parameter'])['value'].mean() computes exactly the same numbers; it just returns them stacked in a single column instead of laid out in a grid

In [15]:
aq.pivot_table(index='site', columns='parameter', values='value', aggfunc='count')

parameter,o3,pm10,pm25
site,,,
CNSI,NaN,NaN,714.0
Goleta,711.0,517.0,734.0
Santa Barbara,734.0,766.0,766.0


### Reading a wide table

In [16]:
means['pm25']

site
CNSI             6.083473
Goleta           6.480926
Santa Barbara    6.172324
Name: pm25, dtype: float64

In [18]:
means.loc['Goleta', 'pm25']

# remember that 'site' is now the index rather than a column

6.4809264305177114

In [20]:
flat = means.reset_index()

flat

# puts the station names back as a regular column

parameter,site,o3,pm10,pm25
0,CNSI,NaN,NaN,6.083473
1,Goleta,0.022470,14.972921,6.480926
2,Santa Barbara,0.019822,17.563969,6.172324


### test your knowledge
Build a wide table with parameter down the rows and site across the columns, which is the transpose of the one above, by swapping the index= and columns= arguments. Then decide which of the two layouts you would put in a report for the Air Pollution Control District, and write a sentence saying why.


In [21]:
aq.pivot_table(index='parameter', columns='site', values='value', aggfunc='count')

site,CNSI,Goleta,Santa Barbara
parameter,,,
o3,NaN,711.0,734.0
pm10,NaN,517.0,766.0
pm25,714.0,734.0,766.0


### From a Series back to a table



In [1]:
aq['site'].value_counts()

NameError: name 'aq' is not defined

In [23]:
counts = aq['site'].value_counts().reset_index()

counts

,site,count
0,Santa Barbara,2266
1,Goleta,1962
2,CNSI,714


In [24]:
counts = counts.rename(columns={'count': 'n_readings'})

counts

,site,n_readings
0,Santa Barbara,2266
1,Goleta,1962
2,CNSI,714


In [26]:
summary = pd.merge(flat, counts, on='site')

summary

,site,o3,pm10,pm25,n_readings
0,CNSI,NaN,NaN,6.083473,714
1,Goleta,0.022470,14.972921,6.480926,1962
2,Santa Barbara,0.019822,17.563969,6.172324,2266


### general format for creating data frame and renaming columns:

counts = df['column'].value_counts().reset_index()

counts = counts.rename(columns={'count': 'a_better_name'})